# CNN + BiLSTM 点云序列识别

这个 notebook 接在 `radar_fft_cube_progress_parallel/` 后面使用。前一步已经把 `adc_data_Raw_0.bin` 经过三次 FFT 处理成逐帧点云文件：

```text
point_cloud_000000.npz
point_cloud_000001.npz
point_cloud_000002.npz
...
```

每个 `.npz` 文件保存一帧雷达点云，核心字段是：

| 字段 | 含义 |
| --- | --- |
| `points` | 结构化点云数组 |
| `frame_index` | 原始 frame 编号 |
| `num_points` | 当前 frame 检测到的点数量 |

`points` 中每个点包含：`range_m`、`velocity_mps`、`angle_deg`、`power_db`、`doppler_bin`、`angle_bin`、`range_bin`。

后续识别思路：

1. 把每帧点云规整成固定数量的点。
2. 用 CNN 提取单帧点云的局部/全局空间特征。
3. 把连续多帧组成时间序列。
4. 用 BiLSTM 捕获前后时间变化。
5. 输出二分类结果：`leaving` 或 `non_leaving`。

## 1. 设备选择：CUDA / MPS / CPU

训练时优先使用 NVIDIA CUDA；如果是 Apple Silicon 机器，则使用 MPS；都不可用时回退到 CPU。

## 1.1 存储格式选择：分离式 NPZ 与合并式 HDF5

前面的并行 FFT 流程把每个 frame 保存成一个独立的 `point_cloud_XXXXXX.npz`。在 SSD 上，这种分离式处理通常效率很高，因为：

- 每个 worker 可以独立写一个 frame，天然适合多进程切分；
- 某个 frame 处理失败不会影响其他 frame；
- 后续可以按 frame range 增量补跑，不需要重写一个大文件；
- 数据组织直观，方便检查单帧点云质量。

如果数据放在 HDD 上，小文件随机读取的 seek 开销会更明显。此时可以把所有 `.npz` 合并成一个 HDF5 文件，把点云数组压在一起，训练时按连续 frame 读取。HDF5 使用 `gzip` 压缩即可，`compression_opts=1` 已经足够快，压缩率和速度比较平衡。

推荐策略：

| 场景 | 推荐格式 | 原因 |
| --- | --- | --- |
| SSD + 多进程 FFT 输出 | 分离式 `.npz` | 写入并行、容易切分、容易增量处理 |
| HDD + 大量训练读取 | 合并式 `.h5` | 减少小文件随机读取开销，连续读取更稳定 |


## 1.2 可选：把逐帧 NPZ 合并成 HDF5

下面代码直接读取 `point_cloud_XXXXXX.npz`，把每帧规整成固定大小数组，然后写入一个 HDF5：

```text
radar_point_clouds.h5
├── points        [num_frames, points_per_frame, num_features]
├── frame_index   [num_frames]
└── num_points    [num_frames]
```

如果没有安装 `h5py`，先安装后再运行这一节。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import csv
import math
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = choose_device()
print("Using device:", DEVICE)


## 2. 训练配置

这里的配置和前一步输出格式对齐：输入目录指向并行 FFT 产生的 `point_cloud_*.npz` 文件。每个训练样本由连续 `sequence_length` 帧组成。

In [ ]:
@dataclass
class TrainConfig:
    # 并行 FFT 输出目录，里面存放 point_cloud_XXXXXX.npz。
    point_cloud_dir: Path = Path("radar_fft_cube_progress_parallel/outputs/point_clouds")

    # 标签文件。推荐格式见下一节。
    labels_csv: Path = Path("radar_fft_cube_progress_parallel/outputs/labels.csv")

    # 每个动作样本包含多少帧。论文里的 LSTM 使用 30 个 point clouds，这里默认保持一致。
    sequence_length: int = 30

    # 每帧固定采样多少点。PointNet 常用 2048；CNN 这里也使用 2048 便于统一。
    points_per_frame: int = 2048

    # 每个点使用的特征。字段名必须存在于 point_cloud_XXXXXX.npz 的 points 结构化数组中。
    point_features: tuple[str, ...] = (
        "range_m",
        "velocity_mps",
        "angle_deg",
        "power_db",
        "doppler_bin",
        "angle_bin",
        "range_bin",
    )

    # 模型与训练参数。
    batch_size: int = 8
    num_epochs: int = 30
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    validation_ratio: float = 0.2
    random_seed: int = 42

    # 类别顺序：输出 0 表示 non_leaving，输出 1 表示 leaving。
    class_names: tuple[str, ...] = ("non_leaving", "leaving")


cfg = TrainConfig()
cfg


## 3. 标签文件格式

FFT 并行流程只负责生成逐帧点云，不负责动作标签。CNN + BiLSTM 训练需要一个 `labels.csv` 来告诉模型哪些连续帧属于 leaving，哪些属于 non-leaving。

推荐格式：

```csv
sample_id,start_frame,end_frame,label
s000001,0,29,leaving
s000002,30,59,non_leaving
s000003,60,89,leaving
```

说明：

- `start_frame` 和 `end_frame` 是闭区间。
- 每一行表示一个动作片段。
- 如果片段帧数超过 `sequence_length`，会按开头截取；如果不足，会重复最后一帧补齐。
- `label` 必须是 `non_leaving` 或 `leaving`。

In [ ]:
def load_label_rows(labels_csv: Path) -> list[dict]:
    if not labels_csv.exists():
        raise FileNotFoundError(
            f"Label file not found: {labels_csv}. "
            "Create labels.csv with columns: sample_id,start_frame,end_frame,label."
        )

    rows: list[dict] = []
    with labels_csv.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        required = {"sample_id", "start_frame", "end_frame", "label"}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"labels.csv missing columns: {sorted(missing)}")

        for row in reader:
            rows.append(
                {
                    "sample_id": row["sample_id"],
                    "start_frame": int(row["start_frame"]),
                    "end_frame": int(row["end_frame"]),
                    "label": row["label"].strip(),
                }
            )
    return rows


## 4. 读取 `.npz` 点云并规整成固定大小

每帧点云的点数不固定。CNN 输入需要固定张量，所以这里做：

- 点数多于 `points_per_frame`：按 `power_db` 优先保留强反射点；
- 点数少于 `points_per_frame`：重复采样补齐；
- 点数为 0：用全 0 点云补齐，保证 batch 不崩。

In [ ]:
def point_cloud_path(point_cloud_dir: Path, frame_index: int) -> Path:
    return point_cloud_dir / f"point_cloud_{frame_index:06d}.npz"


def structured_points_to_array(points: np.ndarray, feature_names: Iterable[str]) -> np.ndarray:
    arrays = []
    for name in feature_names:
        if name not in points.dtype.names:
            raise KeyError(f"Feature {name!r} not found in point dtype: {points.dtype.names}")
        arrays.append(points[name].astype(np.float32))
    return np.stack(arrays, axis=1) if arrays else np.empty((len(points), 0), dtype=np.float32)


def sample_or_pad_points(frame_points: np.ndarray, target_points: int) -> np.ndarray:
    num_points, num_features = frame_points.shape
    if num_points == target_points:
        return frame_points

    if num_points == 0:
        return np.zeros((target_points, num_features), dtype=np.float32)

    if num_points > target_points:
        # power_db 是第 4 个特征，强反射点通常更稳定。
        power_col = 3
        keep = np.argsort(frame_points[:, power_col])[-target_points:]
        return frame_points[keep].astype(np.float32)

    extra = target_points - num_points
    repeat_idx = np.random.choice(num_points, size=extra, replace=True)
    padded = np.concatenate([frame_points, frame_points[repeat_idx]], axis=0)
    return padded.astype(np.float32)


def load_one_frame(path: Path, cfg: TrainConfig) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(f"Point cloud file not found: {path}")

    with np.load(path, allow_pickle=False) as data:
        points = data["points"]

    frame_points = structured_points_to_array(points, cfg.point_features)
    frame_points = sample_or_pad_points(frame_points, cfg.points_per_frame)
    return frame_points


## 5. 特征归一化

点云特征的量纲不同：距离是米，速度是米每秒，角度是度，功率是 dB，FFT bin 是整数。训练前需要归一化，否则大数值字段会主导梯度。

## 4.1 可选代码：NPZ 合并为 HDF5

这一节是可选优化。SSD 上可以继续直接读 `.npz`；HDD 或超大规模训练时，可以先合并成 `.h5` 再训练。

In [ ]:
def merge_npz_point_clouds_to_h5(
    point_cloud_dir: Path,
    output_h5: Path,
    cfg: TrainConfig,
    compression_level: int = 1,
) -> Path:
    """Merge per-frame point_cloud_XXXXXX.npz files into one HDF5 file.

    HDF5 is useful when the storage medium has high random-seek cost. We use
    gzip level 1 because it is fast and still reduces file size.
    """
    try:
        import h5py
    except ImportError as exc:
        raise ImportError("h5py is required for HDF5 export. Install h5py first.") from exc

    npz_files = sorted(point_cloud_dir.glob("point_cloud_*.npz"))
    if not npz_files:
        raise FileNotFoundError(f"No point_cloud_*.npz files found in {point_cloud_dir}")

    output_h5.parent.mkdir(parents=True, exist_ok=True)
    num_frames = len(npz_files)
    num_features = len(cfg.point_features)

    with h5py.File(output_h5, "w") as h5:
        points_ds = h5.create_dataset(
            "points",
            shape=(num_frames, cfg.points_per_frame, num_features),
            dtype="float32",
            chunks=(1, cfg.points_per_frame, num_features),
            compression="gzip",
            compression_opts=compression_level,
            shuffle=True,
        )
        frame_index_ds = h5.create_dataset("frame_index", shape=(num_frames,), dtype="int64")
        num_points_ds = h5.create_dataset("num_points", shape=(num_frames,), dtype="int64")

        h5.attrs["point_features"] = ",".join(cfg.point_features)
        h5.attrs["points_per_frame"] = cfg.points_per_frame
        h5.attrs["compression"] = f"gzip:{compression_level}"

        for out_idx, npz_path in enumerate(npz_files):
            with np.load(npz_path, allow_pickle=False) as data:
                raw_points = data["points"]
                frame_index = int(data["frame_index"])
                num_points = int(data["num_points"])

            frame_points = structured_points_to_array(raw_points, cfg.point_features)
            frame_points = sample_or_pad_points(frame_points, cfg.points_per_frame)

            points_ds[out_idx] = frame_points
            frame_index_ds[out_idx] = frame_index
            num_points_ds[out_idx] = num_points

            if (out_idx + 1) % 1000 == 0:
                print(f"Merged {out_idx + 1}/{num_frames} frames")

    print("HDF5 saved:", output_h5)
    return output_h5


# 使用示例：
# merge_npz_point_clouds_to_h5(
#     point_cloud_dir=cfg.point_cloud_dir,
#     output_h5=Path("radar_fft_cube_progress_parallel/outputs/radar_point_clouds.h5"),
#     cfg=cfg,
#     compression_level=1,
# )


## 4.2 可选代码：从 HDF5 读取连续帧

如果已经合并成 HDF5，训练时可以把 `load_one_frame()` 替换为 HDF5 读取逻辑。下面给一个简单 Dataset，用法和后面的 NPZ Dataset 类似。

In [ ]:
class H5RadarPointSequenceDataset(Dataset):
    def __init__(self, h5_path: Path, rows: list[dict], cfg: TrainConfig, normalizer) -> None:
        self.h5_path = h5_path
        self.rows = rows
        self.cfg = cfg
        self.normalizer = normalizer
        self.class_to_index = {name: i for i, name in enumerate(cfg.class_names)}
        self._h5 = None

    def __len__(self) -> int:
        return len(self.rows)

    def _file(self):
        if self._h5 is None:
            import h5py

            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.rows[index]
        frame_indices = self._frame_indices(row["start_frame"], row["end_frame"])

        h5 = self._file()
        # 逐帧读取而不是 fancy indexing。这样即使短序列补齐时 frame index 重复，h5py 也能稳定工作。
        x = np.stack([h5["points"][frame_index] for frame_index in frame_indices], axis=0).astype(np.float32)
        x = self.normalizer(x).astype(np.float32)

        y = self.class_to_index[row["label"]]
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

    def _frame_indices(self, start_frame: int, end_frame: int) -> list[int]:
        indices = list(range(start_frame, end_frame + 1))
        if len(indices) >= self.cfg.sequence_length:
            return indices[: self.cfg.sequence_length]
        while len(indices) < self.cfg.sequence_length:
            indices.append(indices[-1])
        return indices


In [ ]:
class PointFeatureNormalizer:
    def __init__(self, mean: np.ndarray, std: np.ndarray) -> None:
        self.mean = mean.astype(np.float32)
        self.std = np.maximum(std.astype(np.float32), 1e-6)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return (x - self.mean[None, None, :]) / self.std[None, None, :]


def fit_normalizer(rows: list[dict], cfg: TrainConfig, max_frames: int = 300) -> PointFeatureNormalizer:
    # 只抽样一部分 frame 估计均值方差，避免完整扫描过慢。
    all_frame_indices: list[int] = []
    for row in rows:
        all_frame_indices.extend(range(row["start_frame"], row["end_frame"] + 1))

    random.shuffle(all_frame_indices)
    all_frame_indices = all_frame_indices[:max_frames]

    chunks = []
    for frame_index in all_frame_indices:
        path = point_cloud_path(cfg.point_cloud_dir, frame_index)
        if path.exists():
            chunks.append(load_one_frame(path, cfg))

    if not chunks:
        num_features = len(cfg.point_features)
        return PointFeatureNormalizer(np.zeros(num_features), np.ones(num_features))

    values = np.concatenate(chunks, axis=0)
    return PointFeatureNormalizer(values.mean(axis=0), values.std(axis=0))


## 6. Dataset：连续帧组成一个动作样本

一个样本的形状是：

```text
[sequence_length, points_per_frame, num_features]
```

例如默认配置下是：

```text
[30, 2048, 7]
```

In [ ]:
class RadarPointSequenceDataset(Dataset):
    def __init__(self, rows: list[dict], cfg: TrainConfig, normalizer: PointFeatureNormalizer) -> None:
        self.rows = rows
        self.cfg = cfg
        self.normalizer = normalizer
        self.class_to_index = {name: i for i, name in enumerate(cfg.class_names)}

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.rows[index]
        frame_indices = self._frame_indices(row["start_frame"], row["end_frame"])

        frames = []
        for frame_index in frame_indices:
            frame_path = point_cloud_path(self.cfg.point_cloud_dir, frame_index)
            frames.append(load_one_frame(frame_path, self.cfg))

        x = np.stack(frames, axis=0).astype(np.float32)
        x = self.normalizer(x).astype(np.float32)

        label = row["label"]
        if label not in self.class_to_index:
            raise ValueError(f"Unknown label {label!r}. Expected {self.cfg.class_names}.")
        y = self.class_to_index[label]

        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

    def _frame_indices(self, start_frame: int, end_frame: int) -> list[int]:
        indices = list(range(start_frame, end_frame + 1))
        if len(indices) >= self.cfg.sequence_length:
            return indices[: self.cfg.sequence_length]
        if not indices:
            raise ValueError("Empty frame range in labels.csv")
        while len(indices) < self.cfg.sequence_length:
            indices.append(indices[-1])
        return indices


## 7. 模型结构：CNN + BiLSTM

这里的 CNN 不是图像 CNN，而是对点云做 1D point-wise convolution：

```text
每帧点云 [N points, F features]
  -> Conv1d over points
  -> global max pooling + average pooling
  -> frame embedding
```

然后把每帧 embedding 组成时间序列：

```text
[T frames, frame_embedding]
  -> BiLSTM
  -> classifier
  -> leaving / non_leaving
```

CNN 负责学每一帧点云的空间结构；BiLSTM 负责学起身、转身、远离这些动作的时间变化。

In [ ]:
class FramePointCNN(nn.Module):
    def __init__(self, input_dim: int, embedding_dim: int = 256) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 128, kernel_size=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(128, embedding_dim, kernel_size=1),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch * time, points, features]
        x = x.transpose(1, 2)  # -> [batch * time, features, points]
        x = self.net(x)
        max_pool = torch.max(x, dim=-1).values
        avg_pool = torch.mean(x, dim=-1)
        return torch.cat([max_pool, avg_pool], dim=-1)


class CNNBiLSTMClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int = 2,
        cnn_embedding_dim: int = 256,
        lstm_hidden_dim: int = 128,
        lstm_layers: int = 2,
        dropout: float = 0.35,
    ) -> None:
        super().__init__()
        self.frame_encoder = FramePointCNN(input_dim, cnn_embedding_dim)
        frame_dim = cnn_embedding_dim * 2

        self.temporal_encoder = nn.LSTM(
            input_size=frame_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(lstm_hidden_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, time, points, features]
        batch_size, time_steps, num_points, num_features = x.shape
        x = x.reshape(batch_size * time_steps, num_points, num_features)
        frame_features = self.frame_encoder(x)
        frame_features = frame_features.reshape(batch_size, time_steps, -1)

        temporal_features, _ = self.temporal_encoder(frame_features)

        # BiLSTM 输出每个时间步的双向特征。这里取最后一个时间步作为整段动作表示。
        final_feature = temporal_features[:, -1, :]
        return self.classifier(final_feature)


## 8. 训练与验证函数

MPS 对部分 PyTorch 操作的支持和 CUDA 不完全一样。这里的模型只使用常规 `Conv1d`、`BatchNorm1d`、`LSTM`、`Linear`，适合在 CUDA / MPS / CPU 间切换。

In [ ]:
def move_batch_to_device(batch: tuple[torch.Tensor, torch.Tensor], device: torch.device):
    x, y = batch
    return x.to(device), y.to(device)


def run_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    device: torch.device,
) -> dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch in loader:
        x, y = move_batch_to_device(batch, device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        if is_train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += float(loss.detach().cpu()) * y.size(0)
        total_correct += int((preds == y).sum().detach().cpu())
        total_count += y.size(0)

    return {
        "loss": total_loss / max(total_count, 1),
        "accuracy": total_correct / max(total_count, 1),
    }


## 9. 构建 DataLoader 和模型

这一节会读取 `labels.csv`，按比例划分训练集和验证集，然后创建 CNN + BiLSTM 模型。

In [ ]:
def build_dataloaders(cfg: TrainConfig) -> tuple[DataLoader, DataLoader]:
    random.seed(cfg.random_seed)
    np.random.seed(cfg.random_seed)
    torch.manual_seed(cfg.random_seed)

    rows = load_label_rows(cfg.labels_csv)
    normalizer = fit_normalizer(rows, cfg)
    dataset = RadarPointSequenceDataset(rows, cfg, normalizer)

    if len(dataset) < 2:
        raise ValueError("At least two labeled segments are required for train/validation split.")

    val_size = max(1, int(len(dataset) * cfg.validation_ratio))
    train_size = max(1, len(dataset) - val_size)
    if train_size + val_size > len(dataset):
        train_size = len(dataset) - val_size

    generator = torch.Generator().manual_seed(cfg.random_seed)
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )
    return train_loader, val_loader


def build_model(cfg: TrainConfig) -> CNNBiLSTMClassifier:
    return CNNBiLSTMClassifier(
        input_dim=len(cfg.point_features),
        num_classes=len(cfg.class_names),
    ).to(DEVICE)


## 10. 开始训练

训练会保存验证集 accuracy 最高的权重到 `models/cnn_blstm_best.pt`。

In [ ]:
def train(cfg: TrainConfig) -> CNNBiLSTMClassifier:
    train_loader, val_loader = build_dataloaders(cfg)
    model = build_model(cfg)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    model_dir = Path("models")
    model_dir.mkdir(parents=True, exist_ok=True)
    best_path = model_dir / "cnn_blstm_best.pt"

    best_acc = -math.inf
    history = []

    for epoch in range(1, cfg.num_epochs + 1):
        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        with torch.no_grad():
            val_metrics = run_one_epoch(model, val_loader, criterion, None, DEVICE)

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
        }
        history.append(row)

        print(
            f"Epoch {epoch:03d} | "
            f"train loss {row['train_loss']:.4f} acc {row['train_accuracy']:.4f} | "
            f"val loss {row['val_loss']:.4f} acc {row['val_accuracy']:.4f}"
        )

        if row["val_accuracy"] > best_acc:
            best_acc = row["val_accuracy"]
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": cfg.__dict__,
                    "class_names": cfg.class_names,
                    "history": history,
                },
                best_path,
            )

    print("Best checkpoint:", best_path)
    return model


# 运行训练：
# model = train(cfg)


## 11. 单个动作片段推理

训练完成后，可以用连续帧范围做一次 leaving / non-leaving 推理。

In [ ]:
def predict_segment(
    model: CNNBiLSTMClassifier,
    cfg: TrainConfig,
    normalizer: PointFeatureNormalizer,
    start_frame: int,
    end_frame: int,
) -> dict[str, float | str]:
    row = {"sample_id": "prediction", "start_frame": start_frame, "end_frame": end_frame, "label": cfg.class_names[0]}
    dataset = RadarPointSequenceDataset([row], cfg, normalizer)
    x, _ = dataset[0]
    x = x.unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()

    pred_idx = int(np.argmax(probs))
    return {
        "prediction": cfg.class_names[pred_idx],
        "non_leaving_prob": float(probs[0]),
        "leaving_prob": float(probs[1]),
    }


## 12. 这部分和前面 FFT 结果的关系

前面的 `radar_fft_cube_progress_parallel/` 负责把原始雷达 bin 转成点云：

```text
raw ADC bin -> Range FFT -> Doppler FFT -> Angle FFT -> point_cloud_XXXXXX.npz
```

这个 notebook 负责把点云序列转成动作识别结果：

```text
point_cloud_XXXXXX.npz sequence -> CNN frame encoder -> BiLSTM temporal encoder -> leaving prediction
```

这样整个项目链路就是：

```text
FMCW radar signal
  -> radar FFT cube
  -> point cloud
  -> CNN + BiLSTM
  -> user leaving detection
  -> security decision
```